In [0]:
%sql
-- Step 1A: Create Catalog, Schemas, and Volume
-- Run each section and verify before proceeding

-- ============================================================================
-- SECTION 1: Create Catalog
-- ============================================================================
CREATE CATALOG IF NOT EXISTS drug_evaluation
COMMENT 'Drug Performance Evaluation - ML and Agentic Analysis Project';

-- Verify catalog creation
SELECT 'Catalog created successfully!' as status;
SHOW CATALOGS LIKE 'drug_evaluation';

In [0]:
-- ============================================================================
-- SECTION 2: Create Schemas
-- ============================================================================
USE CATALOG drug_evaluation;

CREATE SCHEMA IF NOT EXISTS landing
COMMENT 'Landing zone for raw CSV files and initial ingestion';

CREATE SCHEMA IF NOT EXISTS bronze
COMMENT 'Bronze layer - raw data with minimal transformations and audit columns';

CREATE SCHEMA IF NOT EXISTS silver
COMMENT 'Silver layer - cleaned, validated, and normalized data';

CREATE SCHEMA IF NOT EXISTS gold
COMMENT 'Gold layer - aggregated and business-ready analytics tables';

-- Verify schemas
SELECT 'Schemas created successfully!' as status;
SHOW SCHEMAS IN drug_evaluation;

In [0]:
# Step 2: Download Kaggle Data and Upload to Volume
# Run each cell separately

# ============================================================================
# CELL 1: Install kagglehub
# ============================================================================
%pip install kagglehub --quiet

In [0]:
# ============================================================================
# CELL 2: Download from Kaggle
# ============================================================================
import kagglehub
from pathlib import Path

# Download dataset from Kaggle
print("Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("thedevastator/drug-performance-evaluation")
print(f"✓ Dataset downloaded to: {path}")

# List downloaded files
import os
print("\nDownloaded files:")
for file in os.listdir(path):
    file_path = os.path.join(path, file)
    if os.path.isfile(file_path):
        size_kb = os.path.getsize(file_path) / 1024
        print(f"  • {file} ({size_kb:.2f} KB)")

In [0]:
# ============================================================================
# CELL 3: Upload to Unity Catalog Volume
# ============================================================================
import shutil

# Define volume path
volume_path = "/Volumes/drug_evaluation/landing/raw_data"

# Copy CSV files to Unity Catalog volume
source_files = [
    (f"{path}/Drug.csv", f"{volume_path}/Drug.csv"),
    (f"{path}/Drug_clean.csv", f"{volume_path}/Drug_clean.csv")
]

print("Copying files to Unity Catalog volume...")
for src, dest in source_files:
    shutil.copy2(src, dest)
    print(f"✓ Copied: {Path(dest).name}")

print(f"\n✅ All files uploaded successfully!")
print(f"Volume location: {volume_path}")

In [0]:
%sql
-- ============================================================================
-- VERIFICATION: Confirm All Infrastructure is Ready
-- ============================================================================

-- Check catalog
SELECT 'Catalog' as object_type, catalog_name as name 
FROM system.information_schema.catalogs 
WHERE catalog_name = 'drug_evaluation'

UNION ALL

-- Check schemas
SELECT 'Schema' as object_type, schema_name as name
FROM system.information_schema.schemata
WHERE catalog_name = 'drug_evaluation' 
  AND schema_name IN ('landing', 'bronze', 'silver', 'gold')

UNION ALL

-- Check volume
SELECT 'Volume' as object_type, volume_name as name
FROM system.information_schema.volumes
WHERE volume_catalog = 'drug_evaluation' 
  AND volume_schema = 'landing'

UNION ALL

-- Check metadata table
SELECT 'Table' as object_type, table_name as name
FROM system.information_schema.tables
WHERE table_catalog = 'drug_evaluation'
  AND table_schema = 'landing'
  AND table_name = 'metadata_tracking';

In [0]:
# Verify files in volume
import os

volume_path = "/Volumes/drug_evaluation/landing/raw_data"

print("📁 Files in Unity Catalog Volume:")
print(f"Location: {volume_path}\n")

for file in os.listdir(volume_path):
    file_path = os.path.join(volume_path, file)
    if os.path.isfile(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"✓ {file} ({size_mb:.2f} MB)")

print("\n✅ Infrastructure setup complete!")

In [0]:
%sql
-- ============================================================================
-- BRONZE LAYER: Raw Data Tables with Audit Columns
-- Preserve original data structure with minimal transformation
-- ============================================================================

USE CATALOG drug_evaluation;
USE SCHEMA bronze;

-- ============================================================================
-- Table 1: Bronze Raw Reviews (2,219 records - detailed reviews)
-- ============================================================================
CREATE TABLE IF NOT EXISTS bronze.drug_reviews_raw (
  -- Original columns from Drug.csv
  condition STRING COMMENT 'Medical condition being treated',
  drug STRING COMMENT 'Drug name or combination',
  indication STRING COMMENT 'Drug usage indication',
  type STRING COMMENT 'Drug type: RX (prescription), OTC (over-the-counter), or RX/OTC',
  reviews STRING COMMENT 'Number of reviews (format: "X Reviews")',
  effective DOUBLE COMMENT 'Effectiveness rating (1-5 scale)',
  ease_of_use DOUBLE COMMENT 'Ease of use rating (1-5 scale)',
  satisfaction DOUBLE COMMENT 'Overall satisfaction rating (1-5 scale)',
  information STRING COMMENT 'Detailed drug information and description',
  
  -- Audit columns
  ingestion_timestamp TIMESTAMP COMMENT 'When record was loaded',
  source_file STRING COMMENT 'Source file name',
  record_id STRING COMMENT 'Unique record identifier (generated)'
)
USING DELTA
COMMENT 'Bronze layer - Raw drug reviews with all original data from Drug.csv';

-- ============================================================================
-- Table 2: Bronze Clean Reviews (685 records - aggregated data)
-- ============================================================================
CREATE TABLE IF NOT EXISTS bronze.drug_reviews_clean (
  -- Original columns from Drug_clean.csv
  condition STRING COMMENT 'Medical condition being treated',
  drug STRING COMMENT 'Drug name or combination',
  ease_of_use DOUBLE COMMENT 'Ease of use rating (1-5 scale)',
  effective DOUBLE COMMENT 'Effectiveness rating (1-5 scale)',
  form STRING COMMENT 'Drug form: Tablet, Liquid (Drink), Cream, Capsule, Liquid (Inject), Other',
  indication STRING COMMENT 'Drug usage indication',
  price DOUBLE COMMENT 'Average retail price in USD',
  reviews DOUBLE COMMENT 'Number of reviews (numeric)',
  satisfaction DOUBLE COMMENT 'Overall satisfaction rating (1-5 scale)',
  type STRING COMMENT 'Drug type: RX, OTC, or RX/OTC',
  
  -- Audit columns
  ingestion_timestamp TIMESTAMP COMMENT 'When record was loaded',
  source_file STRING COMMENT 'Source file name',
  record_id STRING COMMENT 'Unique record identifier (generated)'
)
USING DELTA
COMMENT 'Bronze layer - Clean aggregated drug reviews from Drug_clean.csv';

-- Verify tables created
SHOW TABLES IN drug_evaluation.bronze;

In [0]:
%sql
-- ============================================================================
-- SILVER LAYER: Cleaned & Normalized Dimensional Model
-- Cleaned data with proper types, standardized values, dimensional structure
-- ============================================================================

USE CATALOG drug_evaluation;
USE SCHEMA silver;

-- ============================================================================
-- Dimension 1: Drugs Master
-- ============================================================================
CREATE TABLE IF NOT EXISTS silver.dim_drugs (
  drug_key STRING COMMENT 'Surrogate key (drug_name hash)',
  drug_name STRING COMMENT 'Standardized drug name',
  drug_type STRING COMMENT 'Standardized type: RX, OTC, or RX/OTC',
  drug_form STRING COMMENT 'Drug form: Tablet, Liquid, Cream, Capsule, Injection, Other',
  average_price DOUBLE COMMENT 'Average retail price in USD (from clean dataset)',
  drug_information STRING COMMENT 'Full drug description and usage information',
  
  -- Metadata
  first_seen_date TIMESTAMP COMMENT 'First time drug appeared in data',
  last_updated_date TIMESTAMP COMMENT 'Last update timestamp',
  
  PRIMARY KEY (drug_key)
)
USING DELTA
COMMENT 'Silver dimension - Drug master data with standardized attributes';

-- ============================================================================
-- Dimension 2: Conditions Master
-- ============================================================================
CREATE TABLE IF NOT EXISTS silver.dim_conditions (
  condition_key STRING COMMENT 'Surrogate key (condition_name hash)',
  condition_name STRING COMMENT 'Standardized condition name',
  condition_category STRING COMMENT 'Condition category (e.g., Cardiovascular, Dermatology, Pain)',
  indication_type STRING COMMENT 'Indication type from source data',
  
  -- Statistics
  total_drugs_available INT COMMENT 'Number of drugs treating this condition',
  total_reviews_count BIGINT COMMENT 'Total reviews for this condition',
  
  -- Metadata
  first_seen_date TIMESTAMP COMMENT 'First time condition appeared in data',
  last_updated_date TIMESTAMP COMMENT 'Last update timestamp',
  
  PRIMARY KEY (condition_key)
)
USING DELTA
COMMENT 'Silver dimension - Medical conditions with categorization';

-- ============================================================================
-- Fact Table: Drug Performance Reviews
-- ============================================================================
CREATE TABLE IF NOT EXISTS silver.fact_drug_performance (
  performance_key STRING COMMENT 'Surrogate key (hash of drug_key + condition_key + source)',
  drug_key STRING COMMENT 'Foreign key to dim_drugs',
  condition_key STRING COMMENT 'Foreign key to dim_conditions',
  
  -- Performance metrics
  effectiveness_score DOUBLE COMMENT 'Effectiveness rating (1-5)',
  ease_of_use_score DOUBLE COMMENT 'Ease of use rating (1-5)',
  satisfaction_score DOUBLE COMMENT 'Overall satisfaction rating (1-5)',
  review_count INT COMMENT 'Number of reviews (parsed from string)',
  
  -- Attributes
  drug_form STRING COMMENT 'Form of drug for this record',
  drug_type STRING COMMENT 'Type: RX, OTC, RX/OTC',
  price DOUBLE COMMENT 'Price (if available)',
  
  -- Metadata
  data_source STRING COMMENT 'Source dataset: raw or clean',
  load_timestamp TIMESTAMP COMMENT 'ETL load timestamp',
  
  PRIMARY KEY (performance_key)
  -- Note: Foreign key constraints can be added if needed
)
USING DELTA
COMMENT 'Silver fact - Drug performance metrics by condition with foreign keys to dimensions';

-- Verify tables created
SHOW TABLES IN drug_evaluation.silver;

In [0]:
%sql
-- ============================================================================
-- GOLD LAYER: Business-Ready Analytics Tables
-- Pre-aggregated data optimized for dashboards, ML, and reporting
-- ============================================================================

USE CATALOG drug_evaluation;
USE SCHEMA gold;

-- ============================================================================
-- Analytics 1: Drug Effectiveness Rankings by Condition
-- ============================================================================
CREATE TABLE IF NOT EXISTS gold.drug_effectiveness_ranking (
  condition_name STRING COMMENT 'Medical condition',
  drug_name STRING COMMENT 'Drug name',
  drug_type STRING COMMENT 'RX, OTC, or RX/OTC',
  drug_form STRING COMMENT 'Tablet, Liquid, Cream, Capsule, etc.',
  
  -- Performance metrics
  effectiveness_score DOUBLE COMMENT 'Average effectiveness (1-5)',
  ease_of_use_score DOUBLE COMMENT 'Average ease of use (1-5)',
  satisfaction_score DOUBLE COMMENT 'Average satisfaction (1-5)',
  overall_score DOUBLE COMMENT 'Weighted composite score',
  
  -- Rankings
  effectiveness_rank INT COMMENT 'Rank by effectiveness within condition (1=best)',
  satisfaction_rank INT COMMENT 'Rank by satisfaction within condition (1=best)',
  overall_rank INT COMMENT 'Rank by overall score within condition (1=best)',
  
  -- Volume metrics
  total_reviews INT COMMENT 'Total number of reviews',
  average_price DOUBLE COMMENT 'Average retail price',
  
  -- Metadata
  last_updated TIMESTAMP COMMENT 'Last refresh timestamp'
)
USING DELTA
COMMENT 'Gold analytics - Drug rankings by condition for recommendation engines';

-- ============================================================================
-- Analytics 2: Price-Performance Analysis
-- ============================================================================
CREATE TABLE IF NOT EXISTS gold.price_performance_analysis (
  drug_name STRING COMMENT 'Drug name',
  drug_type STRING COMMENT 'RX, OTC, or RX/OTC',
  
  -- Price segments
  price_tier STRING COMMENT 'Low (<$50), Medium ($50-$150), High (>$150)',
  average_price DOUBLE COMMENT 'Average retail price',
  
  -- Performance metrics
  avg_effectiveness DOUBLE COMMENT 'Average effectiveness across all conditions',
  avg_ease_of_use DOUBLE COMMENT 'Average ease of use',
  avg_satisfaction DOUBLE COMMENT 'Average satisfaction',
  
  -- Value metrics
  value_score DOUBLE COMMENT 'Effectiveness per dollar (effectiveness / price)',
  price_percentile INT COMMENT 'Price percentile (1-100)',
  performance_percentile INT COMMENT 'Performance percentile (1-100)',
  
  -- Volume
  total_reviews INT COMMENT 'Total reviews across all conditions',
  conditions_treated INT COMMENT 'Number of conditions this drug treats',
  
  -- Metadata
  last_updated TIMESTAMP COMMENT 'Last refresh timestamp'
)
USING DELTA
COMMENT 'Gold analytics - Price vs performance analysis for cost-effectiveness insights';

-- ============================================================================
-- Analytics 3: Condition Treatment Summary
-- ============================================================================
CREATE TABLE IF NOT EXISTS gold.condition_treatment_summary (
  condition_name STRING COMMENT 'Medical condition',
  condition_category STRING COMMENT 'Condition category',
  
  -- Drug options
  total_drugs_available INT COMMENT 'Number of treatment options',
  rx_drugs_count INT COMMENT 'Prescription drugs available',
  otc_drugs_count INT COMMENT 'Over-the-counter drugs available',
  
  -- Form distribution
  tablet_options INT COMMENT 'Drugs available as tablets',
  liquid_options INT COMMENT 'Liquid formulations',
  topical_options INT COMMENT 'Creams and topical options',
  injectable_options INT COMMENT 'Injectable options',
  
  -- Performance benchmarks
  avg_effectiveness DOUBLE COMMENT 'Average effectiveness across all drugs',
  avg_satisfaction DOUBLE COMMENT 'Average satisfaction',
  best_drug_name STRING COMMENT 'Highest-ranked drug by overall score',
  best_drug_score DOUBLE COMMENT 'Score of the best drug',
  
  -- Price insights
  min_price DOUBLE COMMENT 'Lowest priced option',
  avg_price DOUBLE COMMENT 'Average treatment cost',
  max_price DOUBLE COMMENT 'Highest priced option',
  
  -- Volume
  total_reviews BIGINT COMMENT 'Total reviews for this condition',
  
  -- Metadata
  last_updated TIMESTAMP COMMENT 'Last refresh timestamp'
)
USING DELTA
COMMENT 'Gold analytics - Condition-level treatment landscape for patient education';

-- ============================================================================
-- Analytics 4: Drug Form Comparison
-- ============================================================================
CREATE TABLE IF NOT EXISTS gold.drug_form_comparison (
  drug_form STRING COMMENT 'Tablet, Liquid (Drink), Cream, Capsule, Liquid (Inject), Other',
  
  -- Performance metrics
  avg_effectiveness DOUBLE COMMENT 'Average effectiveness',
  avg_ease_of_use DOUBLE COMMENT 'Average ease of use',
  avg_satisfaction DOUBLE COMMENT 'Average satisfaction',
  
  -- Market metrics
  drug_count INT COMMENT 'Number of drugs in this form',
  total_reviews INT COMMENT 'Total reviews',
  avg_price DOUBLE COMMENT 'Average price',
  
  -- Condition coverage
  conditions_treated INT COMMENT 'Number of conditions treated',
  most_common_condition STRING COMMENT 'Most frequently treated condition',
  
  -- Type distribution
  rx_percentage DOUBLE COMMENT 'Percentage that are prescription',
  otc_percentage DOUBLE COMMENT 'Percentage over-the-counter',
  
  -- Metadata
  last_updated TIMESTAMP COMMENT 'Last refresh timestamp'
)
USING DELTA
COMMENT 'Gold analytics - Drug form comparison for formulation insights';

-- Verify tables created
SHOW TABLES IN drug_evaluation.gold;

In [0]:
# ============================================================================
# BRONZE LAYER DATA LOAD
# Load raw CSV files from volume to bronze tables
# ============================================================================

from pyspark.sql.functions import col, current_timestamp, md5, concat_ws
from pyspark.sql.types import *

# Define paths
volume_path = "/Volumes/drug_evaluation/landing/raw_data"

print("="*80)
print("BRONZE LAYER DATA LOAD")
print("="*80)

# ============================================================================
# Load 1: Raw Reviews (Drug.csv)
# ============================================================================
print("\n1. Loading drug_reviews_raw from Drug.csv...")

df_raw = spark.read.csv(
    f"{volume_path}/Drug.csv",
    header=True,
    inferSchema=True
)

# Rename columns to match snake_case and add audit columns
df_raw_bronze = (
    df_raw
    .withColumnRenamed("Condition", "condition")
    .withColumnRenamed("Drug", "drug")
    .withColumnRenamed("Indication", "indication")
    .withColumnRenamed("Type", "type")
    .withColumnRenamed("Reviews", "reviews")
    .withColumnRenamed("Effective", "effective")
    .withColumnRenamed("EaseOfUse", "ease_of_use")
    .withColumnRenamed("Satisfaction", "satisfaction")
    .withColumnRenamed("Information", "information")
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("record_id", md5(concat_ws("||", col("condition"), col("drug"), col("indication"))))
)

# Write to bronze table (overwriteSchema to handle type mismatches)
df_raw_bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("drug_evaluation.bronze.drug_reviews_raw")

raw_count = df_raw_bronze.count()
print(f"✓ Loaded {raw_count:,} records to bronze.drug_reviews_raw")

# ============================================================================
# Load 2: Clean Reviews (Drug_clean.csv)
# ============================================================================
print("\n2. Loading drug_reviews_clean from Drug_clean.csv...")

df_clean = spark.read.csv(
    f"{volume_path}/Drug_clean.csv",
    header=True,
    inferSchema=True
)

# Rename columns and add audit columns
df_clean_bronze = (
    df_clean
    .withColumnRenamed("Condition", "condition")
    .withColumnRenamed("Drug", "drug")
    .withColumnRenamed("EaseOfUse", "ease_of_use")
    .withColumnRenamed("Effective", "effective")
    .withColumnRenamed("Form", "form")
    .withColumnRenamed("Indication", "indication")
    .withColumnRenamed("Price", "price")
    .withColumnRenamed("Reviews", "reviews")
    .withColumnRenamed("Satisfaction", "satisfaction")
    .withColumnRenamed("Type", "type")
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("record_id", md5(concat_ws("||", col("condition"), col("drug"))))
)

# Write to bronze table (overwriteSchema to handle type mismatches)
df_clean_bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("drug_evaluation.bronze.drug_reviews_clean")

clean_count = df_clean_bronze.count()
print(f"✓ Loaded {clean_count:,} records to bronze.drug_reviews_clean")

# ============================================================================
# Update metadata tracking
# ============================================================================
print("\n3. Updating metadata tracking...")

metadata_records = [
    ("drug_reviews_raw", "Drug.csv", raw_count, "bronze", "success", "Initial load from Kaggle"),
    ("drug_reviews_clean", "Drug_clean.csv", clean_count, "bronze", "success", "Initial load from Kaggle")
]

from datetime import datetime
metadata_df = spark.createDataFrame(
    [(table, file, count, datetime.now(), layer, status, notes) 
     for table, file, count, layer, status, notes in metadata_records],
    ["table_name", "source_file", "record_count", "load_timestamp", "layer", "status", "notes"]
)

metadata_df.write.mode("append").saveAsTable("drug_evaluation.landing.metadata_tracking")

print("✓ Metadata tracking updated")

# ============================================================================
# Verification
# ============================================================================
print("\n" + "="*80)
print("BRONZE LAYER LOAD COMPLETE")
print("="*80)
print(f"✅ drug_reviews_raw: {raw_count:,} records")
print(f"✅ drug_reviews_clean: {clean_count:,} records")
print(f"✅ Total: {raw_count + clean_count:,} records loaded")

In [0]:
%sql
select * from drug_evaluation.bronze.drug_reviews_clean limit 10

In [0]:
%sql
select * from drug_evaluation.bronze.drug_reviews_raw limit 10

In [0]:
# ============================================================================
# SILVER LAYER DATA TRANSFORMATION & LOAD
# Clean, standardize, and load dimensional model
# ============================================================================

from pyspark.sql.functions import (
    col, trim, upper, regexp_replace, when, current_timestamp, 
    md5, concat_ws, first, count, sum as spark_sum, avg, 
    coalesce, lit, initcap, lower, expr, regexp_extract
)
from pyspark.sql.types import *

print("="*80)
print("SILVER LAYER DATA TRANSFORMATION")
print("="*80)

# ============================================================================
# STEP 1: Load and Clean Bronze Data
# ============================================================================
print("\n1. Loading bronze tables...")

# Load bronze tables
df_raw = spark.table("drug_evaluation.bronze.drug_reviews_raw")
df_clean = spark.table("drug_evaluation.bronze.drug_reviews_clean")

print(f"✓ Loaded {df_raw.count():,} raw records")
print(f"✓ Loaded {df_clean.count():,} clean records")

# ============================================================================
# STEP 2: Data Cleansing & Standardization
# ============================================================================
print("\n2. Cleansing and standardizing data...")

# Clean raw dataset
df_raw_cleaned = (
    df_raw
    # Parse "X Reviews" to numeric (extract only digits, use try_cast for safety)
    .withColumn("review_count", 
                expr("try_cast(regexp_extract(reviews, '^([0-9]+)', 1) as int)"))
    # Clean Type field (remove \r\n and whitespace)
    .withColumn("drug_type_clean", 
                trim(regexp_replace(col("type"), "\\r\\n", "")))
    # Standardize condition names (title case)
    .withColumn("condition_clean", 
                initcap(trim(col("condition"))))
    # Standardize drug names
    .withColumn("drug_clean", 
                trim(col("drug")))
    # Cast ratings to double using try_cast for safety (raw data has malformed values)
    .withColumn("effective_clean", expr("try_cast(effective as double)"))
    .withColumn("ease_of_use_clean", expr("try_cast(ease_of_use as double)"))
    .withColumn("satisfaction_clean", expr("try_cast(satisfaction as double)"))
)

# Clean aggregated dataset (already mostly clean)
df_clean_cleaned = (
    df_clean
    .withColumn("drug_type_clean", 
                when(col("type").isNull(), "Unknown")
                .otherwise(trim(col("type"))))
    .withColumn("condition_clean", 
                initcap(trim(col("condition"))))
    .withColumn("drug_clean", 
                trim(col("drug")))
    .withColumn("review_count", col("reviews").cast("int"))
)

print("✓ Data cleansing complete")

# ============================================================================
# STEP 3: Build Dimension - dim_drugs
# ============================================================================
print("\n3. Building dim_drugs dimension table...")

# Aggregate drug information from both sources
drugs_from_raw = (
    df_raw_cleaned
    .groupBy("drug_clean", "drug_type_clean")
    .agg(
        first("information").alias("drug_information"),
        count("*").alias("mention_count")
    )
    .withColumn("source", lit("raw"))
)


drugs_from_clean = (
    df_clean_cleaned
    .groupBy("drug_clean", "drug_type_clean", "form")
    .agg(
        avg("price").alias("average_price"),
        spark_sum("review_count").alias("total_reviews")
    )
    .withColumn("source", lit("clean"))
)

display(drugs_from_raw)

In [0]:

# Combine and deduplicate
dim_drugs = (
    drugs_from_raw
    .join(
        drugs_from_clean,
        on=["drug_clean", "drug_type_clean"],
        how="full_outer"
    )
    .select(
        md5(col("drug_clean")).alias("drug_key"),
        col("drug_clean").alias("drug_name"),
        coalesce(col("drug_type_clean"), lit("Unknown")).alias("drug_type"),
        coalesce(col("form"), lit("Unknown")).alias("drug_form"),
        col("average_price"),
        col("drug_information"),
        current_timestamp().alias("first_seen_date"),
        current_timestamp().alias("last_updated_date")
    )
    .dropDuplicates(["drug_key"])
)

# Write to silver
dim_drugs.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("drug_evaluation.silver.dim_drugs")

drug_count = dim_drugs.count()
print(f"✓ Loaded {drug_count:,} unique drugs to dim_drugs")

# ============================================================================
# STEP 4: Build Dimension - dim_conditions
# ============================================================================
print("\n4. Building dim_conditions dimension table...")

# Categorize conditions (simple rule-based)
def categorize_condition(condition_name):
    """Simple condition categorization"""
    condition_lower = condition_name.lower() if condition_name else ""
    
    if any(word in condition_lower for word in ["pain", "ache", "arthritis"]):
        return "Pain Management"
    elif any(word in condition_lower for word in ["hypertension", "heart", "cardiovascular"]):
        return "Cardiovascular"
    elif any(word in condition_lower for word in ["dermatitis", "skin", "eczema", "psoriasis"]):
        return "Dermatology"
    elif any(word in condition_lower for word in ["diabetes", "glucose"]):
        return "Metabolic"
    elif any(word in condition_lower for word in ["infection", "bacterial", "viral"]):
        return "Infectious Disease"
    elif any(word in condition_lower for word in ["gout", "uric"]):
        return "Rheumatology"
    elif any(word in condition_lower for word in ["reflux", "gerd", "gastro"]):
        return "Gastrointestinal"
    else:
        return "Other"

from pyspark.sql.functions import udf
categorize_udf = udf(categorize_condition, StringType())

# Aggregate condition statistics (use coalesce to handle nulls from try_cast)
from pyspark.sql.functions import coalesce as sql_coalesce

conditions_combined = (
    df_raw_cleaned
    .select("condition_clean", "drug_clean", sql_coalesce(col("review_count"), lit(0)).alias("review_count"))
    .union(
        df_clean_cleaned.select("condition_clean", "drug_clean", sql_coalesce(col("review_count"), lit(0)).alias("review_count"))
    )
)

dim_conditions = (
    conditions_combined
    .groupBy("condition_clean")
    .agg(
        count("drug_clean").alias("total_drugs_available"),
        spark_sum("review_count").alias("total_reviews_count")
    )
    .select(
        md5(col("condition_clean")).alias("condition_key"),
        col("condition_clean").alias("condition_name"),
        categorize_udf(col("condition_clean")).alias("condition_category"),
        lit("Standard").alias("indication_type"),
        col("total_drugs_available"),
        col("total_reviews_count"),
        current_timestamp().alias("first_seen_date"),
        current_timestamp().alias("last_updated_date")
    )
    .dropDuplicates(["condition_key"])
)

# Write to silver
dim_conditions.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("drug_evaluation.silver.dim_conditions")

condition_count = dim_conditions.count()
print(f"✓ Loaded {condition_count:,} unique conditions to dim_conditions")

# ============================================================================
# STEP 5: Build Fact Table - fact_drug_performance
# ============================================================================
print("\n5. Building fact_drug_performance fact table...")

# Prepare fact records from raw dataset
fact_from_raw = (
    df_raw_cleaned
    .select(
        md5(col("drug_clean")).alias("drug_key"),
        md5(col("condition_clean")).alias("condition_key"),
        col("effective_clean").alias("effectiveness_score"),
        col("ease_of_use_clean").alias("ease_of_use_score"),
        col("satisfaction_clean").alias("satisfaction_score"),
        col("review_count"),
        lit(None).cast("string").alias("drug_form"),
        col("drug_type_clean").alias("drug_type"),
        lit(None).cast("double").alias("price"),
        lit("raw").alias("data_source"),
        col("ingestion_timestamp").alias("load_timestamp")
    )
    .withColumn("performance_key", 
                md5(concat_ws("||", col("drug_key"), col("condition_key"), col("data_source"))))
)

# Prepare fact records from clean dataset
fact_from_clean = (
    df_clean_cleaned
    .select(
        md5(col("drug_clean")).alias("drug_key"),
        md5(col("condition_clean")).alias("condition_key"),
        col("effective").alias("effectiveness_score"),
        col("ease_of_use").alias("ease_of_use_score"),
        col("satisfaction").alias("satisfaction_score"),
        col("review_count"),
        col("form").alias("drug_form"),
        col("drug_type_clean").alias("drug_type"),
        col("price"),
        lit("clean").alias("data_source"),
        col("ingestion_timestamp").alias("load_timestamp")
    )
    .withColumn("performance_key", 
                md5(concat_ws("||", col("drug_key"), col("condition_key"), col("data_source"))))
)

# Union both sources
fact_drug_performance = fact_from_raw.union(fact_from_clean)

# Write to silver
fact_drug_performance.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("drug_evaluation.silver.fact_drug_performance")

fact_count = fact_drug_performance.count()
print(f"✓ Loaded {fact_count:,} performance records to fact_drug_performance")

# ============================================================================
# STEP 6: Update Metadata Tracking
# ============================================================================
print("\n6. Updating metadata tracking...")

from datetime import datetime
metadata_records = [
    ("dim_drugs", "bronze.drug_reviews_raw, bronze.drug_reviews_clean", drug_count, "silver", "success", "Dimension table with unique drugs"),
    ("dim_conditions", "bronze.drug_reviews_raw, bronze.drug_reviews_clean", condition_count, "silver", "success", "Dimension table with medical conditions"),
    ("fact_drug_performance", "bronze.drug_reviews_raw, bronze.drug_reviews_clean", fact_count, "silver", "success", "Fact table with performance metrics")
]

metadata_df = spark.createDataFrame(
    [(table, file, count, datetime.now(), layer, status, notes) 
     for table, file, count, layer, status, notes in metadata_records],
    ["table_name", "source_file", "record_count", "load_timestamp", "layer", "status", "notes"]
)

metadata_df.write.mode("append").saveAsTable("drug_evaluation.landing.metadata_tracking")

print("✓ Metadata tracking updated")

# ============================================================================
# VERIFICATION
# ============================================================================
print("\n" + "="*80)
print("SILVER LAYER TRANSFORMATION COMPLETE")
print("="*80)
print(f"✅ dim_drugs: {drug_count:,} unique drugs")
print(f"✅ dim_conditions: {condition_count:,} unique conditions")
print(f"✅ fact_drug_performance: {fact_count:,} performance records")
print(f"✅ Total: {drug_count + condition_count + fact_count:,} records in silver layer")

In [0]:
%sql
select * from drug_evaluation.silver.dim_conditions limit 10

In [0]:
%sql
select * from drug_evaluation.silver.dim_drugs limit 10

In [0]:
%sql
select * from drug_evaluation.silver.fact_drug_performance limit 10

In [0]:
# ============================================================================
# GOLD LAYER DATA TRANSFORMATION & LOAD
# Build business-ready analytics tables from silver dimensional model
# ============================================================================

from pyspark.sql.functions import (
    col, count, avg, sum as spark_sum, min as spark_min, max as spark_max,
    when, lit, current_timestamp, row_number, rank, dense_rank,
    round as spark_round, coalesce, expr, desc
)
from pyspark.sql.window import Window
from datetime import datetime

print("="*80)
print("GOLD LAYER ANALYTICS TRANSFORMATION")
print("="*80)

# ============================================================================
# STEP 1: Load Silver Layer Tables
# ============================================================================
print("\n1. Loading silver layer tables...")

dim_drugs = spark.table("drug_evaluation.silver.dim_drugs")
dim_conditions = spark.table("drug_evaluation.silver.dim_conditions")
fact_performance = spark.table("drug_evaluation.silver.fact_drug_performance")

print(f"✓ Loaded {dim_drugs.count():,} drugs")
print(f"✓ Loaded {dim_conditions.count():,} conditions")
print(f"✓ Loaded {fact_performance.count():,} performance records")

# ============================================================================
# STEP 2: Analytics 1 - Drug Effectiveness Rankings by Condition
# ============================================================================
print("\n2. Building drug_effectiveness_ranking analytics...")

# Join facts with dimensions (use aliases to avoid ambiguous references)
performance_enriched = (
    fact_performance.alias("fact")
    .join(dim_drugs.alias("drug"), col("fact.drug_key") == col("drug.drug_key"), "inner")
    .join(dim_conditions.alias("cond"), col("fact.condition_key") == col("cond.condition_key"), "inner")
    .select(
        col("cond.condition_name"),
        col("cond.condition_category"),
        col("drug.drug_name"),
        col("drug.drug_type"),
        col("drug.drug_form"),
        col("drug.average_price").alias("drug_avg_price"),
        col("fact.effectiveness_score"),
        col("fact.ease_of_use_score"),
        col("fact.satisfaction_score"),
        col("fact.review_count"),
        col("fact.price")
    )
)

# Aggregate by condition and drug
drug_condition_agg = (
    performance_enriched
    .groupBy(
        col("condition_name"),
        col("drug_name"),
        col("drug_type"),
        col("drug_form")
    )
    .agg(
        avg(col("effectiveness_score")).alias("effectiveness_score"),
        avg(col("ease_of_use_score")).alias("ease_of_use_score"),
        avg(col("satisfaction_score")).alias("satisfaction_score"),
        spark_sum(coalesce(col("review_count"), lit(0))).alias("total_reviews"),
        avg(coalesce(col("price"), col("drug_avg_price"))).alias("average_price")
    )
    # Calculate weighted overall score (effectiveness: 40%, satisfaction: 40%, ease of use: 20%)
    .withColumn("overall_score",
                spark_round(
                    (coalesce(col("effectiveness_score"), lit(0)) * 0.4 +
                     coalesce(col("satisfaction_score"), lit(0)) * 0.4 +
                     coalesce(col("ease_of_use_score"), lit(0)) * 0.2),
                    2
                ))
)

# Add rankings within each condition
window_effectiveness = Window.partitionBy("condition_name").orderBy(desc("effectiveness_score"))
window_satisfaction = Window.partitionBy("condition_name").orderBy(desc("satisfaction_score"))
window_overall = Window.partitionBy("condition_name").orderBy(desc("overall_score"))

drug_effectiveness_ranking = (
    drug_condition_agg
    .withColumn("effectiveness_rank", dense_rank().over(window_effectiveness))
    .withColumn("satisfaction_rank", dense_rank().over(window_satisfaction))
    .withColumn("overall_rank", dense_rank().over(window_overall))
    .withColumn("last_updated", current_timestamp())
    .select(
        "condition_name", "drug_name", "drug_type", "drug_form",
        "effectiveness_score", "ease_of_use_score", "satisfaction_score", "overall_score",
        "effectiveness_rank", "satisfaction_rank", "overall_rank",
        "total_reviews", "average_price", "last_updated"
    )
)

# Write to gold
drug_effectiveness_ranking.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("drug_evaluation.gold.drug_effectiveness_ranking")

ranking_count = drug_effectiveness_ranking.count()
print(f"✓ Loaded {ranking_count:,} drug-condition rankings")

# ============================================================================
# STEP 3: Analytics 2 - Price-Performance Analysis
# ============================================================================
print("\n3. Building price_performance_analysis analytics...")

# Aggregate drug performance across all conditions
drug_overall_performance = (
    performance_enriched
    .groupBy("drug_name", "drug_type")
    .agg(
        avg(col("effectiveness_score")).alias("avg_effectiveness"),
        avg(col("ease_of_use_score")).alias("avg_ease_of_use"),
        avg(col("satisfaction_score")).alias("avg_satisfaction"),
        avg(coalesce(col("price"), col("drug_avg_price"))).alias("average_price"),
        spark_sum(coalesce(col("review_count"), lit(0))).alias("total_reviews"),
        count("condition_name").alias("conditions_treated")
    )
    .filter(col("average_price").isNotNull())  # Only drugs with price data
)

# Add price tiers
price_performance = (
    drug_overall_performance
    .withColumn("price_tier",
                when(col("average_price") < 50, "Low (<$50)")
                .when(col("average_price") < 150, "Medium ($50-$150)")
                .otherwise("High (>$150)"))
    # Calculate value score (effectiveness per dollar)
    .withColumn("value_score",
                spark_round(
                    when(col("average_price") > 0,
                         coalesce(col("avg_effectiveness"), lit(0)) / col("average_price") * 100)
                    .otherwise(0),
                    4
                ))
)

# Calculate percentiles for price and performance
window_price = Window.orderBy("average_price")
window_performance = Window.orderBy(desc("avg_effectiveness"))

price_performance_analysis = (
    price_performance
    .withColumn("price_percentile",
                spark_round((row_number().over(window_price) / count("*").over(Window.partitionBy()) * 100), 0).cast("int"))
    .withColumn("performance_percentile",
                spark_round((row_number().over(window_performance) / count("*").over(Window.partitionBy()) * 100), 0).cast("int"))
    .withColumn("last_updated", current_timestamp())
    .select(
        "drug_name", "drug_type", "price_tier", "average_price",
        "avg_effectiveness", "avg_ease_of_use", "avg_satisfaction",
        "value_score", "price_percentile", "performance_percentile",
        "total_reviews", "conditions_treated", "last_updated"
    )
)

# Write to gold
price_performance_analysis.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("drug_evaluation.gold.price_performance_analysis")

price_count = price_performance_analysis.count()
print(f"✓ Loaded {price_count:,} price-performance records")

# ============================================================================
# STEP 4: Analytics 3 - Condition Treatment Summary
# ============================================================================
print("\n4. Building condition_treatment_summary analytics...")

# Aggregate condition-level statistics
condition_summary = (
    performance_enriched
    .groupBy("condition_name", "condition_category")
    .agg(
        count("drug_name").alias("total_drugs_available"),
        spark_sum(when(col("drug_type") == "RX", 1).otherwise(0)).alias("rx_drugs_count"),
        spark_sum(when(col("drug_type").isin(["OTC", "RX/OTC"]), 1).otherwise(0)).alias("otc_drugs_count"),
        spark_sum(when(col("drug_form").isin(["Tablet", "Capsule"]), 1).otherwise(0)).alias("tablet_options"),
        spark_sum(when(col("drug_form") == "Liquid", 1).otherwise(0)).alias("liquid_options"),
        spark_sum(when(col("drug_form").isin(["Cream", "Gel", "Ointment"]), 1).otherwise(0)).alias("topical_options"),
        spark_sum(when(col("drug_form") == "Injectable", 1).otherwise(0)).alias("injectable_options"),
        avg(col("effectiveness_score")).alias("avg_effectiveness"),
        avg(col("satisfaction_score")).alias("avg_satisfaction"),
        spark_min(coalesce(col("price"), col("drug_avg_price"))).alias("min_price"),
        spark_max(coalesce(col("price"), col("drug_avg_price"))).alias("max_price"),
        avg(coalesce(col("price"), col("drug_avg_price"))).alias("avg_price"),
        spark_sum(coalesce(col("review_count"), lit(0))).alias("total_patient_reviews")
    )
)

# Get best drug per condition (by overall score)
window_best_drug = Window.partitionBy("condition_name").orderBy(desc("effectiveness_score"))

best_drugs = (
    performance_enriched
    .withColumn("rank", row_number().over(window_best_drug))
    .filter(col("rank") == 1)
    .select(
        col("condition_name"),
        col("drug_name").alias("best_drug_by_effectiveness")
    )
)

# Join with best drug
condition_treatment_summary = (
    condition_summary
    .join(best_drugs, "condition_name", "left")
    .withColumn("last_updated", current_timestamp())
    .select(
        "condition_name", "condition_category",
        "total_drugs_available", "rx_drugs_count", "otc_drugs_count",
        "tablet_options", "liquid_options", "topical_options", "injectable_options",
        "avg_effectiveness", "avg_satisfaction",
        "best_drug_by_effectiveness",
        "min_price", "max_price", "avg_price",
        "total_patient_reviews",
        "last_updated"
    )
)

# Write to gold
condition_treatment_summary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("drug_evaluation.gold.condition_treatment_summary")

condition_count = condition_treatment_summary.count()
print(f"✓ Loaded {condition_count:,} condition summaries")

# ============================================================================
# STEP 5: Update Metadata Tracking
# ============================================================================
print("\n5. Updating metadata tracking...")

metadata_records = [
    ("drug_effectiveness_ranking", "silver.fact_drug_performance, silver.dim_drugs, silver.dim_conditions", ranking_count, "gold", "success", "Drug rankings by condition with scores and ranks"),
    ("price_performance_analysis", "silver.fact_drug_performance, silver.dim_drugs", price_count, "gold", "success", "Price vs performance analysis with value scores"),
    ("condition_treatment_summary", "silver.fact_drug_performance, silver.dim_conditions, silver.dim_drugs", condition_count, "gold", "success", "Condition-level treatment options and benchmarks")
]

metadata_df = spark.createDataFrame(
    [(table, file, count, datetime.now(), layer, status, notes) 
     for table, file, count, layer, status, notes in metadata_records],
    ["table_name", "source_file", "record_count", "load_timestamp", "layer", "status", "notes"]
)

metadata_df.write.mode("append").saveAsTable("drug_evaluation.landing.metadata_tracking")

print("✓ Metadata tracking updated")

# ============================================================================
# VERIFICATION
# ============================================================================
print("\n" + "="*80)
print("GOLD LAYER TRANSFORMATION COMPLETE")
print("="*80)
print(f"✅ drug_effectiveness_ranking: {ranking_count:,} records")
print(f"✅ price_performance_analysis: {price_count:,} records")
print(f"✅ condition_treatment_summary: {condition_count:,} records")
print(f"✅ Total: {ranking_count + price_count + condition_count:,} records in gold layer")
print("\n🎯 Gold layer analytics ready for dashboards, ML, and reporting!")